# 6주차 실습 — 판다스(Pandas) 따라 하기

2026 컴퓨터코딩 · 데이터 분석 입문

**Google Colab에서 사용하는 방법**
1. [colab.research.google.com](https://colab.research.google.com) 접속(구글 계정 로그인)
2. **파일 → 노트 업로드**에서 이 파일을 열기 (또는 새 노트를 만들고 슬라이드를 보며 직접 따라 입력)
3. 코드 셀을 클릭하고 **Shift + Enter**로 실행 — 결과는 셀 아래에 바로 표시됩니다

> pandas는 Colab에 이미 설치되어 있어 `import`만 하면 됩니다. 만든 노트는 구글 드라이브에 자동 저장됩니다.

각 셀을 **위에서부터 순서대로** 실행해 보세요. 예상 결과는 마크다운 설명과 주석에 함께 적어 두었습니다.

**오늘의 이야기** — 월요일 아침, 첫 주식으로 **CODE**를 주당 10,000원에 샀습니다. 일주일 뒤 금요일 저녁, 판다스로 나의 첫 거래 일주일을 복기해 봅니다.

## 0. 시작하기
`pandas`를 `pd`라는 별명으로 부르는 것은 넘파이의 `np`와 같은 표준 관례입니다. 대부분의 책·강의·예제 코드가 `pd.`로 시작합니다.

In [ ]:
import pandas as pd

pd.__version__   # Colab에는 이미 설치되어 있음 — 버전은 조금씩 다를 수 있음

## 1. Series 만들기 — `pd.Series`

**Series**는 '이름 붙은 한 줄의 데이터'입니다. 왼쪽에 **인덱스**(행 이름), 오른쪽에 값이 나란히 놓인 모양이며, CODE의 주간 종가처럼 한 줄짜리 데이터를 담을 때 씁니다.

```python
closes = pd.Series([10000, 10200, 9800, 10500, 10800])
closes
```

```
0    10000
1    10200
2     9800
3    10500
4    10800
dtype: int64
```

왼쪽의 `0 1 2 3 4`가 인덱스입니다. 값을 지정하지 않으면 위치 번호가 기본 인덱스가 됩니다.

In [ ]:
closes = pd.Series([10000, 10200, 9800, 10500, 10800])
closes

In [ ]:
closes.tolist()    # [10000, 10200, 9800, 10500, 10800] — 파이썬 리스트로
closes.values      # 넘파이 배열! 지난 주에 배운 ndarray가 그 안에 있습니다
type(closes.values)

> 💡 `.values`가 넘파이 배열을 돌려준다는 것 — 판다스가 **넘파이 위에 얹힌 도구**라는 사실을 보여줍니다.

## 2. 인덱스 — 행에 이름 붙이기

`index=` 로 행마다 이름(라벨)을 붙일 수 있습니다. 주가 데이터에는 날짜·요일이 인덱스로 어울립니다.

```python
closes = pd.Series([10000, 10200, 9800, 10500, 10800],
                   index=["월", "화", "수", "목", "금"])
closes
```

```
월    10000
화    10200
수     9800
목    10500
금    10800
dtype: int64
```

In [ ]:
closes = pd.Series([10000, 10200, 9800, 10500, 10800],
                   index=["월", "화", "수", "목", "금"])
closes

In [ ]:
closes["화"]        # 10200 — 라벨 하나로 값 읽기
closes.loc["금"]    # 10800 — .loc 도 라벨로 읽기
closes.loc["화":"목"]   # 라벨 슬라이스 — 화·수·목 (끝 라벨 포함!)

> ⚠ 위치 슬라이스 `closes[0:2]`는 끝을 **제외**하지만, 라벨 슬라이스 `loc["화":"목"]`는 끝 라벨을 **포함**합니다.

## 3. Series 벡터 연산 — 반복문 없이 한 번에

넘파이에서 배운 벡터화 연산이 그대로 통합니다. CODE를 10,000원에 샀으니, `closes - 10000`은 **하루하루의 손익**이 됩니다.

In [ ]:
gain = closes - 10000    # 전체에서 한 번에 — 반복문 없음
gain.tolist()           # [0, 200, -200, 500, 800]

In [ ]:
closes.mean()             # 10260.0 — 주간 평균 종가
closes.max()              # 10800 — 주간 최고
(closes > 10000).sum()    # 3 — True는 1이므로 합계 = '이긴 날'의 수

## 4. 불리언 마스크 — 조건에 맞는 날만 골라내기

비교 연산의 결과는 날마다 True/False인 Series입니다. 이 **마스크**를 대괄호에 넣으면 True인 날만 남습니다 — 넘파이와 완전히 같은 규칙입니다.

In [ ]:
mask = closes > 10000
mask.tolist()      # [False, True, False, True, True]

In [ ]:
closes[mask]          # 살 때보다 비싸게 끝난 날 — 화·목·금
closes[mask].mean()  # 10500.0 — 이긴 날들의 평균

> ⚠ 조건을 조합할 때는 `and`/`or`가 아니라 **`&` / `|`** 를 쓰고, 조건마다 **괄호**로 묶습니다 — 넘파이와 같습니다.
> 예: `(closes > 10000) & (closes < 10600)` → 화·목

## 5. 최고는 언제였나 — `idxmax`

`max()`는 최고 **값**, `idxmax()`는 최고 값의 **인덱스 라벨**을 돌려줍니다. '언제 가장 높았나'라는 질문에 바로 답하는 함수입니다.

In [ ]:
closes.max()             # 10800 — 최고 종가(값)
closes.idxmax()          # '금' — 최고 종가의 날(라벨)
closes[closes.idxmax()]  # 10800 — 그 날의 값

## 6. DataFrame 만들기 — 딕셔너리로 표

**DataFrame**은 행과 열이 있는 표입니다. `{열 이름: 리스트}` 딕셔너리를 `pd.DataFrame()`에 넘기면 됩니다 — 금요일 저녁, 내가 가진 세 종목의 상태입니다.

| ticker | shares | buy | close |
|---|---|---|---|
| CODE | 10 | 10000 | 10800 |
| TEA | 40 | 5000 | 4800 |
| ROBO | 5 | 40000 | 45000 |

`ticker`는 종목 코드, `shares`는 보유 주식 수, `buy`는 산 가격, `close`는 금요일 종가입니다.

In [ ]:
portfolio = pd.DataFrame({
    "ticker": ["CODE", "TEA", "ROBO"],
    "shares": [10, 40, 5],
    "buy":    [10000, 5000, 40000],
    "close":  [10800, 4800, 45000],
})
portfolio

> ⚠ 모든 열의 리스트 **길이가 같아야** 합니다. 하나라도 길이가 다르면 `ValueError`가 납니다.
> 각 열은 하나의 Series이고, 열들이 같은 인덱스(0, 1, 2)를 공유합니다.

## 7. 리스트의 딕셔너리 — 거래 하나가 한 행

딕셔너리의 리스트(레코드 목록)로도 만들 수 있습니다. 거래 하나가 딕셔너리 하나 — 매매 일지에 어울리는 모양입니다. 아래는 이번 주에 남긴 10건의 거래입니다.

In [ ]:
trades = [
    {"ticker": "CODE", "action": "buy",  "shares": 10, "amount": 100000},
    {"ticker": "TEA",  "action": "sell", "shares": 3,  "amount": 15000},
    {"ticker": "TEA",  "action": "buy",  "shares": 10, "amount": 50000},
    {"ticker": "CODE", "action": "sell", "shares": 2,  "amount": 20000},
    {"ticker": "ROBO", "action": "buy",  "shares": 1,  "amount": 40000},
    {"ticker": "CODE", "action": "sell", "shares": 1,  "amount": 10000},
    {"ticker": "TEA",  "action": "sell", "shares": 3,  "amount": 15000},
    {"ticker": "CODE", "action": "buy",  "shares": 2,  "amount": 20000},
    {"ticker": "TEA",  "action": "buy",  "shares": 6,  "amount": 30000},
    {"ticker": "ROBO", "action": "buy",  "shares": 1,  "amount": 40000},
]
log = pd.DataFrame(trades)
log.shape        # (10, 4)

In [ ]:
log.head(3)      # 처음 몇 행만 미리보기

## 8. 표 둘러보기 — `shape` · `head` · `tail` · `columns`

표를 처음 만나면 **몇 행인지, 어떤 열이 있는지**부터 확인합니다 — 나중에 디버깅할 때도 이 네 가지가 첫 단추입니다.

In [ ]:
portfolio.shape            # (3, 4) — (행 수, 열 수)
portfolio.head(2)          # 위에서 두 행
portfolio.tail(1)          # 아래에서 한 행
portfolio.columns.tolist() # ['ticker', 'shares', 'buy', 'close']

## 9. 열 선택 — 한 개는 Series, 여러 개는 DataFrame

대괄호에 **열 이름 문자열** 하나를 넣으면 Series, **이름의 리스트**를 넣으면 DataFrame이 나옵니다. 대괄호 수를 헷갈리지 않게 주의하세요.

In [ ]:
close = portfolio["close"]     # 한 개 — Series
close.tolist()                 # [10800, 4800, 45000]

In [ ]:
friday = portfolio[["ticker", "close"]]   # 리스트 — DataFrame(3, 2)
friday

> ⚠ `portfolio["close"]`는 Series, `portfolio[["close"]]`는 열 하나짜리 DataFrame입니다. `type(...)`으로 직접 확인해 보세요.

## 10. 행 선택 — `iloc`(위치)와 `loc`(라벨)

- `iloc` : **위치 번호**(0부터)로 행을 고름 — 넘파이 인덱싱과 같은 느낌
- `loc`  : **인덱스 라벨**로 행을 고름 — 위의 `closes.loc["화":...]`와 같은 규칙

In [ ]:
portfolio.iloc[0]        # 첫 번째 행 — CODE (결과는 Series)
portfolio.iloc[0:2]      # 슬라이스 — 첫 두 행 (끝 위치 제외)
portfolio.iloc[-1]       # 마지막 행 — ROBO

In [ ]:
portfolio.iloc[-1]["ticker"]   # 'ROBO' — 마지막 행의 종목
closes.loc["화":"목"]          # Series는 라벨로 — 화·수·목 (끝 포함)

## 11. 정렬 — `sort_values`

어느 열을 기준으로 줄 세울지 정합니다. `ascending=False`를 주면 큰 것부터 — **행 전체가 값과 함께** 움직입니다. 정렬 결과는 새 표이고 원본은 그대로입니다.

In [ ]:
ranked = portfolio.sort_values("close", ascending=False)
ranked

In [ ]:
ranked["ticker"].tolist()   # ['ROBO', 'CODE', 'TEA'] — 종가가 높은 순서
portfolio["ticker"].tolist()  # ['CODE', 'TEA', 'ROBO'] — 원본은 그대로!

## 12. 한 열의 모든 통계 — `describe`

`count · mean · std · min · 25% · 50% · 75% · max`를 한 번에 보여줍니다. 표를 처음 받으면 숫자 열에 `describe()`부터 찍어보는 것이 분석의 첫걸음입니다.

In [ ]:
c = portfolio["close"]
c.max(), round(c.mean(), 1)   # (45000, 20200.0)

In [ ]:
c.describe()

## 13. 파생 컬럼 — 계산 결과를 새 열로

`df["새 열 이름"] = 계산식` 처럼 대입하면 새 열이 생깁니다. 벡터 연산이 **모든 행에 한 번에** 적용됩니다 — 반복문이 필요 없습니다.

`invested`(투자금) = 보유량 × 매수가, `profit`(평가손익) = (종가 − 매수가) × 보유량

In [ ]:
portfolio["invested"] = portfolio["shares"] * portfolio["buy"]
portfolio["profit"] = (portfolio["close"] - portfolio["buy"]) * portfolio["shares"]
portfolio

In [ ]:
portfolio["profit"].tolist()   # [8000, -8000, 25000] — TEA는 8,000원 손실

## 14. 값 세기 — `value_counts`

열에서 값이 각각 몇 번 나오는지 세어 **많은 것부터** 보여줍니다. 문자열 열에도 쓸 수 있어, 매매 일지의 매수/매도 횟수 같은 질문에 바로 답합니다.

In [ ]:
counts = log["action"].value_counts()
counts          # buy 6건, sell 4건

In [ ]:
counts.index.tolist()   # ['buy', 'sell'] — 많은 순서대로
log["amount"].sum()    # 340000 — 이번 주 총 거래액

## 15. 행 필터링 — 마스크로 행 골라내기

열에 대한 조건(마스크)을 표 전체의 대괄호에 넣으면 **True인 행만** 남습니다. 조건을 조합할 때는 `&`, `|` 와 괄호를 씁니다. 결과를 새 변수에 담으면 원본은 지켜집니다.

In [ ]:
winners = portfolio[portfolio["profit"] > 0]
winners[["ticker", "profit"]]

In [ ]:
winners.shape    # (2, 6) — 이긴 종목 2개 (열은 파생 컬럼까지 6개)
big = log[log["amount"] >= 40000]
big["action"].tolist()   # ['buy', 'buy', 'buy', 'buy'] — 큰 거래는 전부 매수였다!

## 16. 종합 예제 — 나의 첫 주식 거래 일주일

오늘 만든 세 가지 데이터로 한 주를 복기합니다 — ① CODE 종가(Series) ② 금요일 포트폴리오(DataFrame) ③ 매매 일지(DataFrame).

In [ ]:
print("1막 — CODE 주간 종가")
print("하루하루 손익:", (closes - 10000).tolist())
print("주간 평균:", closes.mean())
print("최고의 날:", closes.idxmax(), closes.max())
print("이긴 날:", (closes > 10000).sum(), "일")

In [ ]:
print("2막 — 금요일 저녁 포트폴리오")
print(portfolio.sort_values("profit", ascending=False)[["ticker", "profit"]])
print("이긴 종목:", portfolio[portfolio["profit"] > 0]["ticker"].tolist())
print("총 투자금:", portfolio["invested"].sum(), "원")

In [ ]:
print("3막 — 매매 일지")
print(log["action"].value_counts().to_dict())
print("총 거래액:", log["amount"].sum(), "원")
print("큰 거래(4만 원 이상):", (log["amount"] >= 40000).sum(), "건 — 전부 매수")

> 💡 **감 잡기(선택)**: `closes`와 `portfolio`의 숫자를 본인 가상 포트폴리오로 바꿔서 다시 실행해 보세요. ` Series → DataFrame → 요약`의 흐름이 손에 익습니다.

## 오늘의 정리
1. **Series**는 '이름 붙은 한 열' — 인덱스가 행 라벨, `.values`는 넘파이 배열
2. **만들기**: `pd.Series(리스트)` · `pd.DataFrame({열: 리스트})` · 레코드(딕셔너리) 리스트 — 모든 열의 길이가 같아야 함
3. **선택**: `df["열"]` 한 개(Series) · `df[["열1", "열2"]]` 여러 개(DataFrame) · `iloc` 위치 · `loc` 라벨
4. **계산**: 벡터 연산이 그대로 통하고 대입하면 **파생 컬럼** — 반복문 없이 열 전체가 한 번에
5. **요약**: `sort_values` · `describe` · `idxmax` · `value_counts` · 마스크로 행 필터링

---
### 다음 주 (7주차) 예고 — 맷플롯립(Matplotlib)
오늘 계산한 종가와 수익을 **그래프**로 그려 봅니다. 질문은 LMS 게시판을 이용해 주세요.